# SAM + BAMS Deposition Log

Registra la deposición de SAM (PFBT) y BAMS (semiconductor).

**Proceso:**
1. UVO (UV Ozone) treatment
2. SAM immersion (PFBT)
3. BAMS deposition (semiconductor)

Este notebook crea una memoria de tipo `experiment` con `subtype=session`.

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '../..')))
sys.path.insert(0, str(PROJECT_ROOT))

from src import db

db.init_db()
print('Database initialized')

## 1. Seleccionar Wafer/Piezas

Selecciona el wafer o piezas que vas a procesar.

In [ ]:
# Listar memorias de tipo wafer disponibles
wafers = db.list_memories(type='experiment', subtype='wafer')

if not wafers:
    print('No wafers found. Run template_wafer_fab.ipynb first.')
else:
    print(f'Available wafers ({len(wafers)}):')
    for w in wafers:
        print(f'  [{w["id"]}] {w["title"]} ({w["date"]}) - {w["substrate"]}')

In [ ]:
# === MODIFICAR ESTOS VALORES ===
WAFER_MEMORY_ID = 1  # ID del wafer a procesar
PIECES_TO_PROCESS = ['P1', 'P2']  # Piezas a procesar (ej: ['P1'], ['P1', 'P2', 'P3'])
DATE = datetime.now().strftime('%Y-%m-%d')
# ==============================

# Obtener información del wafer
wafer = db.get_memory(WAFER_MEMORY_ID)
print(f'Processing wafer: {wafer["title"]}')
print(f'Pieces: {", ".join(PIECES_TO_PROCESS)}')

## 2. Crear memoria de sesión SAM+BAMS

In [ ]:
# Crear memoria para esta sesión
session_title = f'SAM+BAMS {DATE} - {wafer["title"]}'

memory_id = db.create_memory(
    title=session_title,
    type='experiment',
    subtype='session',
    date=DATE,
    description=f'Deposición SAM+BAMS en piezas {", ".join(PIECES_TO_PROCESS)} del wafer {wafer["title"]}',
    substrate=wafer['substrate'],
    process='SAM + BAMS',
    custom_fields={
        'wafer_id': str(WAFER_MEMORY_ID),
        'pieces': ','.join(PIECES_TO_PROCESS),
    }
)

print(f'Created session memory with id={memory_id}')

## 3. UVO + SAM

In [ ]:
# === MODIFICAR ESTOS VALORES ===
SAM_PARAMS = {
    'sam_type': 'PFBT',
    'concentration_ul_ml': 2,
    'ipa_volume_ml': 20,
    'pfbt_volume_ul': 40,
    'uvo_time_min': 25,
    'immersion_time_min': 15,
}
# ==============================

# Agregar parámetros SAM como campos
for key, value in SAM_PARAMS.items():
    db.add_field(memory_id, f'sam_{key}', str(value))

print(f'Registered SAM parameters:')
for k, v in SAM_PARAMS.items():
    print(f'  {k}: {v}')

## 4. BAMS Deposition

In [ ]:
# === MODIFICAR ESTOS VALORES ===
BAMS_PARAMS = {
    'bar_number': 3,
    'temp_C': 105,
    'speed_mm_s': 10,
    'motor_steps_s': 2778,
    'volume_ul_per_device': 35,
    'ink_batch': '',  # Llenar con el nombre del ink batch si aplica
}
BAMS_NOTES = ''
# ==============================

# Agregar parámetros BAMS como campos
for key, value in BAMS_PARAMS.items():
    db.add_field(memory_id, f'bams_{key}', str(value))

if BAMS_NOTES:
    db.add_field(memory_id, 'bams_notes', BAMS_NOTES)

print(f'Registered BAMS parameters:')
for k, v in BAMS_PARAMS.items():
    print(f'  {k}: {v}')

## 5. Fotos (opcional)

In [ ]:
# Buscar fotos recientes
photos_dir = PROJECT_ROOT / 'data' / 'photos'

if photos_dir.exists():
    photos = []
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        photos.extend(photos_dir.rglob(ext))
    
    # Ordenar por fecha de modificación (más recientes primero)
    photos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    
    print(f'Found {len(photos)} photos')
    
    # Referenciar primeras 3 fotos
    for photo in photos[:3]:
        rel_path = photo.relative_to(PROJECT_ROOT)
        db.add_reference(memory_id, 'photo', str(rel_path))
        print(f'  Added: {rel_path.name}')
else:
    print('No photos directory found')

## Resumen

In [ ]:
# Mostrar memoria completa
memory = db.get_memory(memory_id)

print(f'=== SAM+BAMS Session {memory_id} ===')
print(f'Title: {memory["title"]}')
print(f'Date: {memory["date"]}')
print(f'Substrate: {memory["substrate"]}')

print(f'\nFields ({len(memory["fields"])}):')
for f in memory['fields']:
    if f['field_type'] == 'custom':
        print(f'  - {f["field_name"]}: {f["field_value"]}')

print(f'\nReferences ({len(memory["references"])}):')
for r in memory['references']:
    print(f'  - {r["ref_type"]}: {r["ref_path"]}')

---
### Notas
- Almacenar dispositivos en vacuum/glovebox después de BAMS
- Continuar con `template_measurement.ipynb` para registrar mediciones